# Where does the time go? Explicit GPT-2 vs CODI

**Kaggle: Settings → Accelerator → GPU T4 x2; Internet on; Run All.** One T4 is used.
No attachments, old summaries, settings changes, or prior fitted heads are needed.

The final output is one table: **Explicit | CODI overall | CODI latent only | CODI visible only**.
Every value is mean **milliseconds per question**, with the total at the top and bottom.
The rows cover all 12 transformer blocks, the head, transfers, text preparation, and runtime overhead.

This runs the PDF's **rank-96 global head, eager PyTorch, FP16, batch 1**. Both paths use
CODI's released GPT-2 weights: explicit generates the teacher-style reasoning text;
CODI takes six latent steps before its visible answer. Each mode gets its own fitted head.
Nested 32/64/96 prefixes remain part of the training recipe; there is no deployment ablation sweep.

In [ ]:
import gc,gzip,hashlib,importlib.util,json,logging,os,pathlib,random,subprocess,sys,time,warnings
from contextlib import contextmanager
from urllib.request import urlopen

# Fixed experiment defaults: no configuration is required.
SEED=89
MODES=['codi','explicit_cot']
FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS=1024,256,256
MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES=4096,1024,2048
CLEAN_EPOCHS,RECOVERY_EPOCHS=4,2
DISTILL_BATCH_SIZE=8
COLLECT_BATCH_SIZE=8
RANKS=(32,64,96)
MAX_NEW_TOKENS={'codi':64,'explicit_cot':256}
TIMING_QUESTIONS,TIMING_REPEATS=16,3
OUTPUT_ROOT=pathlib.Path('/kaggle/working/codi_bottleneck')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
BOOTSTRAP_TIMES=[]
@contextmanager
def bootstrap_stage(name):
    start=time.perf_counter()
    try: yield
    finally: BOOTSTRAP_TIMES.append(dict(name=name,cpu_wall_ms=1000*(time.perf_counter()-start)))

# Do not reinstall datasets/pandas/dill or change Kaggle's PyTorch/CUDA.
# GSM8K is read directly from its canonical JSONL files.
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ.pop('HF_HUB_DISABLE_XET',None)
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT','300')
REPO_URL='https://github.com/0x0shephard/latent-reasoning.git'
BASE_COMMIT='6a8d2e61950c67f012d0a9ba13ec8a70f3a25019'
REPO_DIR='/kaggle/working/latent-reasoning'
setup_log=OUTPUT_ROOT/'setup.log'
def checked(command):
    result=subprocess.run(command,capture_output=True,text=True)
    with setup_log.open('a') as log: log.write(result.stdout+'\n'+result.stderr+'\n')
    if result.returncode:
        raise RuntimeError(result.stdout[-3000:]+'\n'+result.stderr[-5000:])
    return result
with bootstrap_stage('git_clone_and_checkout'):
    if not pathlib.Path(REPO_DIR).exists(): checked(['git','clone',REPO_URL,REPO_DIR])
    checked(['git','-C',REPO_DIR,'fetch','origin'])
    checked(['git','-C',REPO_DIR,'checkout','--detach',BASE_COMMIT])
os.chdir(REPO_DIR)
sys.path.insert(0,REPO_DIR)
with bootstrap_stage('dependency_installation'):
    checked([sys.executable,'-m','pip','install','-q','transformers==4.52.4','peft==0.15.2',
             'huggingface_hub>=0.34.0,<1.0','hf_xet','accelerate==1.7.0','pyyaml'])
    probe=subprocess.run([sys.executable,'-c','import peft; from transformers import GPT2LMHeadModel'],capture_output=True,text=True)
    if probe.returncode and 'torchao' in (probe.stdout+probe.stderr):
        checked([sys.executable,'-m','pip','uninstall','-y','torchao'])
    checked([sys.executable,'-c','import torch,peft,huggingface_hub,hf_xet; from transformers import GPT2LMHeadModel,DynamicCache'])
print('Setup ready. Unrelated system-package messages are retained in setup.log; failed installs/imports stop here.')

In [ ]:
RUNTIME_SOURCE = '"""CODI/explicit decoding and an additive, four-column bottleneck report.\n\nThe report partitions a single CUDA-stream timeline, including host-induced idle\nintervals. It is an instrumented elapsed-time breakdown, not summed kernel time.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nimport gzip\nimport json\nfrom pathlib import Path\nimport time\nimport uuid\n\nimport torch\nfrom torch import nn\nfrom src.models.official_codi import official_codi_base_model, _normalized_official_questions\nfrom src.inference.official_codi_fast import FastCODIGeneration\n\n\nclass Timeline:\n    def __init__(self, device=\'cpu\', enabled=True, unified_clock=False):\n        self.device = torch.device(device)\n        self.enabled = enabled\n        self.unified_clock = unified_clock\n        self.context = {}\n        self.records = []\n        self.pending = []\n        self.stack = []\n        self.counter = 0\n        self.trace_id = uuid.uuid4().hex\n\n    def start(self, name, kind=\'stage\', gpu=True, **extra):\n        if not self.enabled:\n            return None\n        self.counter += 1\n        row = dict(self.context, trace_id=self.trace_id, event_id=self.counter,\n                   parent_id=self.stack[-1] if self.stack else None,\n                   name=name, kind=kind, **extra)\n        marker = torch.profiler.record_function(name)\n        marker.__enter__()\n        row[\'_wall_start\'] = time.perf_counter()\n        events = None\n        if (gpu or self.unified_clock) and self.device.type == \'cuda\':\n            events = (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))\n            events[0].record()\n        self.stack.append(self.counter)\n        return row, events, marker\n\n    def stop(self, token):\n        if token is None:\n            return\n        row, events, marker = token\n        if events:\n            events[1].record()\n        row[\'cpu_wall_ms\'] = (time.perf_counter() - row.pop(\'_wall_start\')) * 1000\n        marker.__exit__(None, None, None)\n        if self.stack and self.stack[-1] == row[\'event_id\']:\n            self.stack.pop()\n        self.pending.append((row, events))\n\n    @contextmanager\n    def span(self, name, kind=\'stage\', gpu=True, **extra):\n        token = self.start(name, kind, gpu, **extra)\n        try:\n            yield\n        finally:\n            self.stop(token)\n\n    @contextmanager\n    def metadata(self, **values):\n        previous = self.context.copy()\n        self.context.update(values)\n        try:\n            yield\n        finally:\n            self.context = previous\n\n    def resolve(self):\n        if self.device.type == \'cuda\' and self.pending:\n            torch.cuda.synchronize(self.device)\n        for row, events in self.pending:\n            row[\'cuda_stream_ms\'] = events[0].elapsed_time(events[1]) if events else None\n            self.records.append(row)\n        self.pending.clear()\n\n    @contextmanager\n    def modules(self, roots, recursive=True):\n        if not self.enabled:\n            yield\n            return\n        handles, stacks, seen = [], {}, set()\n        for prefix, root in roots:\n            for suffix, module in (root.named_modules() if recursive else [(\'\', root)]):\n                if id(module) in seen:\n                    continue\n                seen.add(id(module))\n                name = prefix + (\'.\' + suffix if suffix else \'\')\n                stacks[id(module)] = []\n                def pre(mod, args, label=name):\n                    token = self.start(label, kind=\'module\', inclusive=True,\n                                       module_type=type(mod).__name__)\n                    stacks[id(mod)].append(token)\n                def post(mod, args, output):\n                    self.stop(stacks[id(mod)].pop())\n                handles.append(module.register_forward_pre_hook(pre))\n                handles.append(module.register_forward_hook(post, always_call=True))\n        try:\n            yield\n        finally:\n            for handle in handles:\n                handle.remove()\n\n    def flush(self, path):\n        self.resolve()\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with gzip.open(path, \'at\', encoding=\'utf-8\') as handle:\n            for row in self.records:\n                handle.write(json.dumps(row, default=str) + \'\\n\')\n        self.records.clear()\n\n\n@dataclass\nclass PreparedBatch:\n    indices: tuple\n    ids: torch.Tensor\n    mask: torch.Tensor\n\n\ndef prepare_questions(tokenizer, questions, batch_size, timeline=None):\n    timeline = timeline or Timeline(enabled=False)\n    with timeline.span(\'question_normalization\', gpu=False):\n        questions = _normalized_official_questions(questions)\n    with timeline.span(\'question_tokenization\', gpu=False):\n        encoded = tokenizer(questions, add_special_tokens=False, padding=False)[\'input_ids\']\n    result = []\n    for start in range(0, len(encoded), batch_size):\n        part = encoded[start:start+batch_size]\n        with timeline.metadata(batch_index=start//batch_size):\n            with timeline.span(\'cpu_padding_and_tensor_allocation\', gpu=False):\n                width = max(map(len, part))\n                ids = torch.full((len(part), width), tokenizer.pad_token_id, dtype=torch.long)\n                mask = torch.zeros_like(ids)\n                for i, values in enumerate(part):\n                    if not values:\n                        raise ValueError(\'Empty question\')\n                    ids[i, -len(values):] = torch.tensor(values)\n                    mask[i, -len(values):] = 1\n            result.append(PreparedBatch(tuple(range(start, start+len(part))), ids, mask))\n    return result\n\n\ndef select_token(head, hidden, vocabulary_stop):\n    specialized = getattr(head, \'select_token\', None)\n    return specialized(hidden, vocabulary_stop=vocabulary_stop) if callable(specialized) else head(hidden)[..., :vocabulary_stop].argmax(-1)\n\n\n@torch.no_grad()\ndef decode(model, tokenizer, batches, head, *, mode, device, max_new_tokens=256,\n           latent_iterations=6, timeline=None, observer=None, forced_tokens=None):\n    """Same CODI body-only contract; explicit mode starts directly from the question.\n\n    Fixed replay evaluates every head but feeds back dense-reference token IDs.\n    Timed CUDA paths must be called inside torch.inference_mode() by the runner.\n    """\n    if mode not in (\'codi\', \'explicit_cot\'):\n        raise ValueError(mode)\n    if max_new_tokens <= 0:\n        raise ValueError(\'max_new_tokens must be positive\')\n    timeline = timeline or Timeline(device, enabled=False)\n    device = torch.device(device)\n    base = official_codi_base_model(model)\n    body, embedding = base.transformer, model.input_embeddings()\n    model.eval()\n    head.eval()\n    vocabulary_stop = int(model.eot_id)\n    eos = int(tokenizer.eos_token_id)\n    total = sum(len(b.indices) for b in batches)\n    outputs, texts, counts_out = [None]*total, [None]*total, [0]*total\n    with timeline.span(\'answer_cue_tokenization\', gpu=False):\n        cue_ids = tokenizer(\' The answer is:\', add_special_tokens=False)[\'input_ids\'] if mode == \'codi\' else []\n    for batch_number, batch in enumerate(batches):\n        with timeline.metadata(batch_index=batch_number, question_indices=list(batch.indices), token_position=-1):\n            with timeline.span(\'host_to_device_input_ids\'):\n                ids = batch.ids.to(device, non_blocking=True)\n            with timeline.span(\'host_to_device_attention_mask\'):\n                mask = batch.mask.to(device, non_blocking=True)\n            with timeline.span(\'prompt_tensor_construction\'):\n                if mode == \'codi\':\n                    bot = torch.full((len(ids),1), model.bot_id, device=device, dtype=torch.long)\n                    ids = torch.cat((ids,bot),1)\n                    mask = torch.cat((mask,torch.ones_like(bot)),1)\n            # Do not silently truncate GPT-2 context.\n            maximum = getattr(model.config, \'n_positions\', 1024) if hasattr(model, \'config\') else 1024\n            reserve = latent_iterations + 1 + len(cue_ids) if mode == \'codi\' else 0\n            if ids.shape[1] + reserve + max_new_tokens > maximum:\n                raise ValueError(\'Prompt plus generation exceeds context; reduce the configured token cap\')\n            with timeline.span(\'prefill_position_ids\'):\n                positions = mask.long().cumsum(-1)-1 if mode == \'explicit_cot\' else None\n                if positions is not None:\n                    positions.masked_fill_(mask == 0, 1)\n            with timeline.metadata(phase=\'prefill\'):\n                with timeline.span(\'transformer_prefill\'):\n                    prefill_kwargs = dict(input_ids=ids, attention_mask=mask, use_cache=True, return_dict=True)\n                    if getattr(getattr(body, \'config\', None), \'model_type\', None) == \'gpt2\':\n                        from transformers import DynamicCache\n                        prefill_kwargs[\'past_key_values\'] = DynamicCache()\n                    if positions is not None:\n                        prefill_kwargs[\'position_ids\'] = positions\n                    out = body(**prefill_kwargs)\n            with timeline.span(\'kv_cache_reference_update\'):\n                cache = out.past_key_values\n                hidden = out.last_hidden_state[:, -1:, :]\n            if mode == \'codi\':\n                with timeline.metadata(phase=\'latent\'), timeline.span(\'phase_latent\'):\n                    with timeline.metadata(phase=\'latent_projection_initial\'):\n                        with timeline.span(\'latent_projector\'):\n                            latent = model.prj(hidden)\n                    for step in range(latent_iterations):\n                        with timeline.metadata(phase=\'latent\', latent_step=step):\n                            with timeline.span(\'transformer_latent_pass\'):\n                                out = body(inputs_embeds=latent, past_key_values=cache, use_cache=True, return_dict=True)\n                            with timeline.span(\'kv_cache_reference_update\'):\n                                cache = out.past_key_values\n                            with timeline.span(\'latent_projector\'):\n                                latent = model.prj(out.last_hidden_state[:, -1:, :])\n            with timeline.metadata(phase=\'visible_decode\'), timeline.span(\'phase_visible\'):\n                if mode == \'codi\':\n                    with timeline.metadata(phase=\'answer_cue\'):\n                        with timeline.span(\'answer_cue_tensor_construction\'):\n                            cue = torch.tensor([model.eot_id,*cue_ids], device=device).unsqueeze(0).expand(len(ids),-1)\n                        with timeline.span(\'answer_cue_embedding\'):\n                            cue_embedding = embedding(cue)\n                        with timeline.span(\'transformer_answer_cue\'):\n                            out = body(inputs_embeds=cue_embedding, past_key_values=cache, use_cache=True, return_dict=True)\n                        cache = out.past_key_values\n                        hidden = out.last_hidden_state[:, -1:, :]\n                with timeline.span(\'generation_buffer_allocation\'):\n                    tokens = torch.full((len(ids), max_new_tokens), eos, device=device, dtype=torch.long)\n                    counts = torch.zeros(len(ids), device=device, dtype=torch.long)\n                    finished = torch.zeros(len(ids), device=device, dtype=torch.bool)\n                replay = None\n                if forced_tokens is not None:\n                    with timeline.span(\'replay_cpu_padding\', gpu=False):\n                        seqs = [forced_tokens[i] for i in batch.indices]\n                        limit = max(map(len,seqs))\n                        if limit > max_new_tokens or any(not seq for seq in seqs):\n                            raise ValueError(\'Replay tokens must fit the generation cap\')\n                        replay_cpu = torch.full((len(ids),limit), eos, dtype=torch.long)\n                        for row, seq in enumerate(seqs):\n                            replay_cpu[row,:len(seq)] = torch.tensor(seq)\n                    with timeline.span(\'host_to_device_replay_tokens\'):\n                        replay = replay_cpu.to(device)\n                else:\n                    limit = max_new_tokens\n                for position in range(limit):\n                    with timeline.metadata(phase=\'visible_decode\', token_position=position):\n                        if observer:\n                            with timeline.span(\'collect_hidden_state_to_cpu\'):\n                                observer(hidden[:, -1, :], ~finished, position, batch.indices)\n                        with timeline.span(\'lm_head_and_argmax\'):\n                            predicted = select_token(head, hidden[:, -1, :], vocabulary_stop)\n                        with timeline.span(\'token_selection_and_buffers\'):\n                            token = predicted if replay is None else replay[:, position]\n                            active = ~finished\n                            tokens[:,position] = torch.where(active,token,tokens[:,position])\n                            counts += active.long()\n                            finished |= active & (token == eos)\n                        with timeline.span(\'termination_check_host_sync\'):\n                            stop = position+1 == limit or bool(finished.all())\n                        if stop:\n                            break\n                        with timeline.span(\'next_token_embedding\'):\n                            embedded = embedding(token).unsqueeze(1)\n                        with timeline.span(\'decode_attention_mask_update\'):\n                            if mode == \'explicit_cot\':\n                                mask = torch.cat((mask,torch.ones((len(ids),1),device=device,dtype=mask.dtype)),1)\n                        with timeline.span(\'transformer_visible_token\'):\n                            kwargs = dict(inputs_embeds=embedded, past_key_values=cache, use_cache=True, return_dict=True)\n                            if mode == \'explicit_cot\':\n                                kwargs[\'attention_mask\'] = mask\n                                kwargs[\'position_ids\'] = (mask.long().sum(-1)-1).unsqueeze(1)\n                            out = body(**kwargs)\n                        with timeline.span(\'kv_cache_reference_update\'):\n                            cache, hidden = out.past_key_values, out.last_hidden_state\n                with timeline.span(\'device_to_host_token_buffer\'):\n                    cpu_tokens = tokens.cpu()\n                with timeline.span(\'device_to_host_counts\'):\n                    cpu_counts = counts.cpu().tolist()\n                with timeline.span(\'cpu_token_conversion_and_text_decode\', gpu=False):\n                    for row,index in enumerate(batch.indices):\n                        count = int(cpu_counts[row])\n                        seq = tuple(int(t) for t in cpu_tokens[row,:count].tolist())\n                        outputs[index], counts_out[index] = seq, count\n                        texts[index] = tokenizer.decode(seq, skip_special_tokens=True)\n            if not timeline.unified_clock:\n                timeline.resolve()\n    return FastCODIGeneration(tuple(texts),tuple(outputs),tuple(counts_out))\n\n\nclass DenseSelector(nn.Module):\n    def __init__(self, head, vocabulary_size):\n        super().__init__()\n        self.head = head\n        self.vocabulary_size = vocabulary_size\n    def forward(self, hidden):\n        return self.head(hidden)[..., :self.vocabulary_size]\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\nclass FixedRankHead(nn.Module):\n    def __init__(self, source, rank=96):\n        super().__init__()\n        self.vocabulary_size = source.vocabulary_size\n        self.down = nn.Linear(source.hidden_size,rank)\n        self.up = nn.Linear(rank,source.vocabulary_size)\n        with torch.no_grad():\n            self.down.weight.copy_(source.down.weight[:rank])\n            self.down.bias.copy_(source.down.bias[:rank])\n            self.up.weight.copy_(source.up.weight[:,:rank])\n            self.up.bias.copy_(source.up.bias)\n        self.requires_grad_(False)\n    def forward(self, hidden):\n        return self.up(self.down(hidden))\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\n\nCOLUMNS = (\'Explicit\', \'CODI overall\', \'CODI latent only\', \'CODI visible only\')\nBREAKDOWN_ROWS = (\n    \'Question loading / tokenization\', \'CPU padding / allocation\',\n    \'Input transfer to GPU\', \'Embeddings\',\n    *(f\'Transformer block {i + 1:02d}\' for i in range(12)),\n    \'Transformer final norm\', \'Transformer masks / bookkeeping\',\n    \'Latent projector\', \'Low-rank LM head\', \'Argmax / head dispatch\',\n    \'Token buffers / cache updates\', \'EOS check / synchronization\',\n    \'Output transfer to CPU\', \'Text decoding\', \'Python / tracing gaps\',\n)\n\n\ndef _category(row, ancestors):\n    # Descendant events inherit their enclosing component. Summing exclusive\n    # durations reconstructs that component without counting nested modules twice.\n    for event in [row, *ancestors]:\n        name = event[\'name\']\n        if name.startswith(\'transformer.h.\'):\n            return f"Transformer block {int(name.split(\'.\')[2]) + 1:02d}"\n        if name.startswith(\'transformer.ln_f\'):\n            return \'Transformer final norm\'\n        if name.startswith((\'transformer.wte\', \'transformer.wpe\')):\n            return \'Embeddings\'\n        if name.startswith(\'projector\'):\n            return \'Latent projector\'\n        if name.startswith(\'lm_head\') and name != \'lm_head_and_argmax\':\n            return \'Low-rank LM head\'\n    name = row[\'name\']\n    if name in (\'load_question\', \'question_normalization\', \'question_tokenization\', \'answer_cue_tokenization\'):\n        return \'Question loading / tokenization\'\n    if name == \'cpu_padding_and_tensor_allocation\':\n        return \'CPU padding / allocation\'\n    if name.startswith(\'host_to_device\'):\n        return \'Input transfer to GPU\'\n    if name in (\'next_token_embedding\', \'answer_cue_embedding\'):\n        return \'Embeddings\'\n    if name.startswith(\'transformer_\') or name in (\'prefill_position_ids\', \'decode_attention_mask_update\'):\n        return \'Transformer masks / bookkeeping\'\n    if name == \'latent_projector\':\n        return \'Latent projector\'\n    if name == \'lm_head_and_argmax\':\n        return \'Argmax / head dispatch\'\n    if name == \'termination_check_host_sync\':\n        return \'EOS check / synchronization\'\n    if name.startswith(\'device_to_host\'):\n        return \'Output transfer to CPU\'\n    if name == \'cpu_token_conversion_and_text_decode\':\n        return \'Text decoding\'\n    if name in (\'kv_cache_reference_update\', \'generation_buffer_allocation\',\n                \'token_selection_and_buffers\', \'prompt_tensor_construction\', \'answer_cue_tensor_construction\'):\n        return \'Token buffers / cache updates\'\n    return \'Python / tracing gaps\'\n\n\ndef partition_question(events):\n    """Exclusive intervals on one clock; never add CPU and CUDA measurements."""\n    by_id = {r[\'event_id\']: r for r in events}\n    roots = [r for r in events if r[\'name\'] == \'question_total\']\n    if len(roots) != 1:\n        raise ValueError(\'Expected exactly one complete question trace\')\n    root = roots[0]\n    clock = \'cuda_stream_ms\' if root.get(\'cuda_stream_ms\') is not None else \'cpu_wall_ms\'\n    if any(r.get(clock) is None for r in events):\n        raise ValueError(\'Every span must use the same clock; enable unified_clock\')\n    children = {}\n    for row in events:\n        children.setdefault(row.get(\'parent_id\'), []).append(row)\n    values = {}\n    for row in events:\n        exclusive = row[clock] - sum(c[clock] for c in children.get(row[\'event_id\'], []))\n        if exclusive < -0.01:\n            raise ValueError(f"Overlapping timing spans: {row[\'name\']}: {exclusive} ms")\n        # Keep sub-microsecond event rounding differences so totals reconcile.\n        ancestors = []\n        parent = row.get(\'parent_id\')\n        while parent is not None:\n            ancestors.append(by_id[parent]); parent = by_id[parent].get(\'parent_id\')\n        category = _category(row, ancestors)\n        phase = row.get(\'phase\', \'shared\')\n        phase = (\'latent\' if phase in (\'latent\', \'latent_projection_initial\') else\n                 \'visible\' if phase in (\'visible_decode\', \'answer_cue\') else \'shared\')\n        key = (category, phase)\n        values[key] = values.get(key, 0.0) + exclusive\n    return dict(mode=root[\'mode\'], question_id=root.get(\'question_id\'), repeat=root.get(\'repeat\'),\n                total_ms=root[clock], clock=clock,\n                breakdown=[dict(name=name, phase=phase, ms=value) for (name, phase), value in values.items()])\n\n\ndef bottleneck_means(samples):\n    """Four data columns, means per question (including zero-work questions)."""\n    groups = {mode: [s for s in samples if s[\'mode\'] == mode] for mode in (\'explicit_cot\', \'codi\')}\n    if any(not group for group in groups.values()):\n        raise ValueError(\'Both reasoning modes need timing samples\')\n    rows = {name: dict.fromkeys(COLUMNS, 0.0) for name in BREAKDOWN_ROWS}\n    totals = dict.fromkeys(COLUMNS, 0.0)\n    for mode, group in groups.items():\n        overall = \'Explicit\' if mode == \'explicit_cot\' else \'CODI overall\'\n        for sample in group:\n            totals[overall] += sample[\'total_ms\'] / len(group)\n            for item in sample[\'breakdown\']:\n                value = item[\'ms\'] / len(group)\n                rows[item[\'name\']][overall] += value\n                if mode == \'codi\' and item[\'phase\'] in (\'latent\', \'visible\'):\n                    column = \'CODI latent only\' if item[\'phase\'] == \'latent\' else \'CODI visible only\'\n                    rows[item[\'name\']][column] += value\n                    totals[column] += value\n    return [(\'Total average time\', totals), *rows.items(), (\'Total average time (repeat)\', dict(totals))]\n\n\n@torch.inference_mode()\ndef profile_question(model, tokenizer, head, question, *, mode, device, max_new_tokens,\n                     question_id=0, repeat=0):\n    """One batch-1 sample, retaining every module call for optional inspection."""\n    device = torch.device(device)\n    timeline = Timeline(device, unified_clock=True)\n    timeline.context.update(mode=mode, question_id=question_id, repeat=repeat, phase=\'shared\')\n    base = official_codi_base_model(model)\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n    with timeline.modules([(\'transformer\', base.transformer), (\'projector\', model.prj), (\'lm_head\', head)]):\n        with timeline.span(\'question_total\'):\n            with timeline.span(\'load_question\', gpu=False):\n                questions = [str(question)]\n            batches = prepare_questions(tokenizer, questions, 1, timeline)\n            result = decode(model, tokenizer, batches, head, mode=mode, device=device,\n                            max_new_tokens=max_new_tokens, timeline=timeline)\n    timeline.resolve()\n    sample = partition_question(timeline.records)\n    sample.update(tokens=list(result.token_ids[0]), text=result.texts[0],\n                  visible_tokens=result.generated_token_counts[0])\n    return sample, timeline.records\n'
EXPERIMENT_SOURCE_SHA256 = '6e47caf99d18a98dee7cd49cf7b054fea327f5ecc2b3b97c2c241d96657cbf77'
runtime_path=pathlib.Path('/kaggle/working/dual_global_head_runtime.py')
runtime_path.write_text(RUNTIME_SOURCE)
spec=importlib.util.spec_from_file_location('dual_global_head_runtime',runtime_path)
runtime=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=runtime
spec.loader.exec_module(runtime)
import torch
import pandas as pd
from dataclasses import asdict
from importlib.metadata import version
from src.mech.global_low_rank_head import (
    NestedLowRankVocabularyHead,activation_whitened_factors,distil_nested_head,evaluate_nested_head)
from src.models.official_codi import (
    build_official_codi_gpt2,download_official_checkpoint,load_official_checkpoint,official_codi_base_model)
from src.inference.official_codi_fast import (
    generate_official_codi_fast,prepare_official_codi_batches,merge_official_codi_lora_)
from src.utils.config import load_config
assert torch.cuda.is_available(),'Select Settings > Accelerator > GPU T4 x2.'
assert 'T4' in torch.cuda.get_device_name(0),'Select GPU T4 x2 in Kaggle settings and restart the session.'
device=torch.device('cuda:0')
DEPLOY_DTYPE=torch.float16
torch.manual_seed(SEED); random.seed(SEED)
config=dict(seed=SEED,fit=FIT_QUESTIONS,selection=SELECT_QUESTIONS,recovery=RECOVERY_QUESTIONS,
    state_caps=[MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES],ranks=RANKS,
    epochs=[CLEAN_EPOCHS,RECOVERY_EPOCHS],token_caps=MAX_NEW_TOKENS,
    questions=TIMING_QUESTIONS,repeats=TIMING_REPEATS,batch_size=1,head='rank96_eager',
    base=BASE_COMMIT,source=EXPERIMENT_SOURCE_SHA256,torch=torch.__version__,cuda=torch.version.cuda,
    packages={name:version(name) for name in ('transformers','peft','huggingface_hub','hf_xet','accelerate')},
    gpu=torch.cuda.get_device_name(0))
RUN_DIR=OUTPUT_ROOT/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR.mkdir(parents=True,exist_ok=True)
DEBUG_PATH=RUN_DIR/'debug.jsonl.gz'
# Each Run All starts a fresh measurement log; fitted heads in the same run folder are reused.
with gzip.open(DEBUG_PATH,'wt') as stream: pass

def debug_record(kind,payload):
    with gzip.open(DEBUG_PATH,'at',encoding='utf-8') as stream:
        stream.write(json.dumps(dict(kind=kind,data=payload),default=str)+'\n')
def save_json(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); temp.write_text(json.dumps(value,indent=2,default=str)); temp.replace(path)
def save_pt(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); torch.save(value,temp); temp.replace(path)
setup=runtime.Timeline(device)
setup.context.update(mode='setup')
setup.records.extend(BOOTSTRAP_TIMES)
def flush_setup():
    setup.resolve()
    for row in setup.records: debug_record('setup',row)
    setup.records.clear()
debug_record('manifest',config)
debug_record('dependency_log',setup_log.read_text())
flush_setup()
# These two construction notices are expected in the pinned official loader.
# Preserve them in the debug log and keep unrelated warnings visible.
class KnownConstructionNotice(logging.Filter):
    def filter(self,record):
        if record.getMessage().startswith('The new embeddings will be initialized'):
            debug_record('construction_notice',record.getMessage()); return False
        return True
logging.getLogger('transformers.modeling_utils').addFilter(KnownConstructionNotice())
print(f"Using {config['gpu']}; rank 96, FP16, batch 1. Fitting is the slow setup step.")

Download the official checkpoint and fit on GSM8K train. Timing questions are held out from fitting.

In [ ]:
DATA_REVISION='3101c7d5072418e28b9008a6636bde82a006892c'
url=f'https://raw.githubusercontent.com/openai/grade-school-math/{DATA_REVISION}/grade_school_math/data/train.jsonl'
with setup.span('gsm8k_download_parse_and_partition',gpu=False):
    with urlopen(url,timeout=300) as response:
        train=[json.loads(line) for line in response if line.strip()]
    unique={}
    for row in train:
        key=' '.join(row['question'].casefold().split())
        unique.setdefault(key,dict(question=str(row['question']),gold=str(row['answer'])))
    rows=list(unique.values()); random.Random(SEED).shuffle(rows)
    splits={}; start=0
    for name,size in zip(['fit','selection','recovery','timing','warmup'],
                         [FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS,TIMING_QUESTIONS,4]):
        splits[name]=rows[start:start+size]; start+=size
    assert start<=len(rows)
    debug_record('partitions',splits)
with setup.span('config_load',gpu=False): cfg=load_config('configs/official_codi_gpt2.yaml')
with setup.span('checkpoint_download_and_hash',gpu=False):
    checkpoint=download_official_checkpoint(repo_id=cfg.checkpoint.repo_id,revision=cfg.checkpoint.revision,
        filename=cfg.checkpoint.filename,expected_sha256=cfg.checkpoint.sha256)
with setup.span('gpt2_model_and_tokenizer_load',gpu=False):
    with warnings.catch_warnings(record=True) as notices:
        warnings.filterwarnings('always',message='fan_in_fan_out is set to False.*')
        model,tokenizer=build_official_codi_gpt2(base_model=cfg.model.base_model,base_revision=cfg.model.base_revision,
            dtype=torch.float32,settings=cfg.model)
    for notice in notices:
        if 'fan_in_fan_out is set to False' in str(notice.message):
            debug_record('construction_notice',str(notice.message))
        else: warnings.warn(str(notice.message),notice.category)
with setup.span('checkpoint_load_and_verification',gpu=False):
    load_report=load_official_checkpoint(model,checkpoint,expected_sha256=cfg.checkpoint.sha256)
    debug_record('checkpoint',asdict(load_report))
with setup.span('model_host_to_device_fp32'): model.requires_grad_(False).to(device).eval()
base=official_codi_base_model(model)
full_head=base.get_output_embeddings()
weight=full_head.weight[:model.eot_id].detach()
bias=None if getattr(full_head,'bias',None) is None else full_head.bias[:model.eot_id].detach()
flush_setup()

In [ ]:
parity_questions=[r['question'] for r in splits['selection'][:4]]
parity={}
with torch.no_grad():
    for mode in MODES:
        prepared=runtime.prepare_questions(tokenizer,parity_questions,4)
        observed=runtime.decode(model,tokenizer,prepared,full_head,mode=mode,device=device,
            max_new_tokens=MAX_NEW_TOKENS[mode],latent_iterations=6)
        if mode=='codi':
            reference=generate_official_codi_fast(model,tokenizer,
                prepare_official_codi_batches(tokenizer,parity_questions,batch_size=4,length_bucketed=False),
                latent_iterations=6,max_new_tokens=MAX_NEW_TOKENS[mode],device=device,answer_cue='The answer is:')
            expected=reference.token_ids
        else:
            from transformers import LogitsProcessor,LogitsProcessorList
            class VocabularyBoundary(LogitsProcessor):
                def __call__(self,input_ids,scores):
                    scores[:,int(model.eot_id):]=float('-inf'); return scores
            batch=prepared[0]
            generated=base.generate(input_ids=batch.ids.to(device),attention_mask=batch.mask.to(device),
                do_sample=False,max_new_tokens=MAX_NEW_TOKENS[mode],pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,logits_processor=LogitsProcessorList([VocabularyBoundary()]))
            expected=[]
            for seq in generated[:,batch.ids.shape[1]:].cpu().tolist():
                if tokenizer.eos_token_id in seq: seq=seq[:seq.index(tokenizer.eos_token_id)+1]
                expected.append(tuple(seq))
            expected=tuple(expected)
        parity[mode]=dict(examples=len(expected),exact=observed.token_ids==expected)
        assert parity[mode]['exact'],f'{mode}: custom decoder differs from reference; stop before fitting'
debug_record('decoder_parity',parity)
print('Decoder checks passed: CODI and explicit GPT-2.')

Fit the proposal’s head for each mode: four clean epochs, then two recovery epochs.

In [ ]:
def collect(mode,head,population,cap,tag):
    path=RUN_DIR/f'states_{mode}_{tag}.pt'
    if path.exists():
        with setup.span('state_cache_disk_load',gpu=False,mode=mode,tag=tag):
            return torch.load(path,map_location='cpu',weights_only=False)
    chunks=[]; positions=[]
    def observe(hidden,active,position,indices):
        chunks.append(hidden[active].detach().cpu().float())
        positions.extend([position]*int(active.sum()))
    questions=[r['question'] for r in splits[population]]
    batches=runtime.prepare_questions(tokenizer,questions,COLLECT_BATCH_SIZE)
    with setup.span('trajectory_collection',mode=mode,population=population):
        runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,
                       max_new_tokens=MAX_NEW_TOKENS[mode],observer=observe)
    values=torch.cat(chunks)
    indices=torch.randperm(len(values),generator=torch.Generator().manual_seed(SEED))[:cap]
    result=dict(states=values[indices].clone(),positions=torch.tensor(positions)[indices],observed_states=len(values))
    with setup.span('state_cache_disk_save',gpu=False,mode=mode,tag=tag): save_pt(path,result)
    return result

def evaluate_positions(head,bundle):
    result={}
    for rank in RANKS:
        result[str(rank)]={}
        for label,mask in [('all',torch.ones(len(bundle['states']),dtype=torch.bool)),
                           ('p0',bundle['positions']==0),('p1',bundle['positions']==1),('p2plus',bundle['positions']>=2)]:
            result[str(rank)][label]=dict(states=int(mask.sum()),**(evaluate_nested_head(
                head,bundle['states'][mask],weight,readout_bias=bias,rank=rank,batch_size=DISTILL_BATCH_SIZE)
                if mask.any() else {}))
    return result

for mode in MODES:
    artifact=RUN_DIR/f'global_head_{mode}.pt'
    if artifact.exists(): print('Reuse fitted head:',mode); continue
    print('Collect/fitting:',mode,flush=True)
    fit=collect(mode,full_head,'fit',MAX_FIT_STATES,'fit')
    selection=collect(mode,full_head,'selection',MAX_SELECT_STATES,'selection')
    with setup.span('activation_whitened_initialization',mode=mode):
        centre,down,up,out_bias,init=activation_whitened_factors(fit['states'],weight,96,
            readout_bias=bias,seed=SEED,compute_device=device)
        head=NestedLowRankVocabularyHead.from_whitened_factors(centre,down,up,out_bias,RANKS).to(device)
    initial_metrics=evaluate_positions(head,selection)
    with setup.span('clean_distillation',mode=mode):
        clean=distil_nested_head(head,fit['states'],selection['states'],weight,readout_bias=bias,
            epochs=CLEAN_EPOCHS,batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED)
    head.disable_adaptive(); head.set_rank(64)
    recovery=collect(mode,head,'recovery',MAX_RECOVERY_STATES,'onpolicy')
    with setup.span('recovery_distillation',mode=mode):
        recovered=distil_nested_head(head,torch.cat((fit['states'],recovery['states'])),
            selection['states'],weight,readout_bias=bias,epochs=RECOVERY_EPOCHS,
            batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED+1)
    report=dict(mode=mode,initialization=asdict(init),clean=asdict(clean),recovery=asdict(recovered),
                initial=initial_metrics,final=evaluate_positions(head,selection),
                fit_states=len(fit['states']),recovery_states=len(recovery['states']))
    with setup.span('trained_head_disk_save',gpu=False,mode=mode):
        save_pt(artifact,dict(state_dict={k:v.detach().cpu().clone() for k,v in head.state_dict().items()},report=report))
    flush_setup()
    del head,fit,selection,recovery,centre,down,up,out_bias
    gc.collect(); torch.cuda.empty_cache()
with setup.span('lora_merge'): merge_official_codi_lora_(model)
with setup.span('model_fp16_conversion'): model.to(dtype=DEPLOY_DTYPE).eval()
base=official_codi_base_model(model); full_head=base.get_output_embeddings()
del weight,bias
flush_setup()

Measure batch-1 generation on the same 16 held-out questions, repeated three times.
Only rank 96 receives detailed profiling. Dense generation supplies a brief unprofiled reference.

In [ ]:
# Timing runner: only one deployed method, one batch size, and two reasoning modes.
@torch.inference_mode()
def clean_generation(mode,head,question):
    if device.type=='cuda': torch.cuda.synchronize(device)
    start=time.perf_counter()
    batches=runtime.prepare_questions(tokenizer,[question],1)
    result=runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode])
    if device.type=='cuda': torch.cuda.synchronize(device)
    return result,1000*(time.perf_counter()-start)

heads={}
for mode in MODES:
    with setup.span('fitted_head_load_and_move',mode=mode):
        payload=torch.load(RUN_DIR/f'global_head_{mode}.pt',map_location='cpu',weights_only=False)
        nested=NestedLowRankVocabularyHead(int(model.config.hidden_size),int(model.eot_id),RANKS)
        nested.load_state_dict(payload['state_dict'])
        heads[mode]=runtime.FixedRankHead(nested,96).to(device=device,dtype=DEPLOY_DTYPE).eval()
        agreement=payload['report']['final']['96']['all']['top1_agreement']
        print(f'{mode}: fitted head validation token agreement {agreement:.1%}.')
        debug_record('fit_report',payload['report'])
    del payload,nested

dense=runtime.DenseSelector(full_head,int(model.eot_id))
with setup.span('generation_warmup'):
    for mode in MODES:
        for row in splits['warmup']:
            for head in (dense,heads[mode]): clean_generation(mode,head,row['question'])
        # Warm the instrumented path too; discard this trace.
        runtime.profile_question(model,tokenizer,heads[mode],splits['warmup'][0]['question'],
            mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode])
flush_setup()

clean_samples=[]
profile_samples=[]
rng=random.Random(SEED)
for repeat in range(TIMING_REPEATS):
    # Finish clean measurements before attaching any profiling hooks.
    jobs=[(mode,i) for mode in MODES for i in range(len(splits['timing']))]
    rng.shuffle(jobs)
    expected={}
    for mode,i in jobs:
        arms=[('dense',dense),('rank96',heads[mode])]; rng.shuffle(arms)
        for arm,head in arms:
            result,elapsed=clean_generation(mode,head,splits['timing'][i]['question'])
            row=dict(mode=mode,arm=arm,question_id=i,repeat=repeat,ms=elapsed,
                     visible_tokens=result.generated_token_counts[0],tokens=list(result.token_ids[0]),text=result.texts[0])
            clean_samples.append(row)
            debug_record('clean_sample',row)
            if arm=='rank96': expected[mode,i]=row['tokens']
    for mode,i in jobs:
        sample,events=runtime.profile_question(model,tokenizer,heads[mode],splits['timing'][i]['question'],
            mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode],question_id=i,repeat=repeat)
        assert sample['tokens']==expected[mode,i],'Instrumentation changed generated tokens'
        profile_samples.append(sample)
        debug_record('profile_sample',sample)
        debug_record('events',events)
        del events
    print(f'Timing repeat {repeat+1}/{TIMING_REPEATS} complete.',flush=True)

**Reading the table:** mean ms/question, using instrumented elapsed time on one CUDA stream.
Parent/child intervals are partitioned so each column adds to its total. Host-induced GPU
idle time is included; these are not pure kernel execution times. Profiling itself adds overhead.

CODI latent = its six reasoning passes and projector; visible = answer cue, answer tokens,
and output handling. Overall also includes shared prompt loading, tokenization, transfer,
and prefill. Downloads and head fitting are one-time setup costs, outside the per-question total.

In [ ]:
# The single main result table: exactly four numeric columns, means only.
report=runtime.bottleneck_means(profile_samples)
bottleneck=pd.DataFrame([values for _,values in report],index=[name for name,_ in report],columns=runtime.COLUMNS)
bottleneck.index.name='Mean ms per question'
assert all(abs(bottleneck.iloc[1:-1][c].sum()-bottleneck.iloc[0][c])<0.05 for c in runtime.COLUMNS)
with pd.option_context('display.max_rows',None,'display.max_columns',4,'display.width',160,'display.float_format',lambda x:f'{x:.3f}'):
    display(bottleneck)
# The same table is saved once; individual cases stay in one optional debug file.
bottleneck.to_csv(RUN_DIR/'bottleneck.csv')
for mode,label in [('explicit_cot','Explicit'),('codi','CODI')]:
    means={arm:sum(r['ms'] for r in clean_samples if r['mode']==mode and r['arm']==arm)/
        sum(r['mode']==mode and r['arm']==arm for r in clean_samples) for arm in ('dense','rank96')}
    lengths={arm:sum(r['visible_tokens'] for r in clean_samples if r['mode']==mode and r['arm']==arm)/
        sum(r['mode']==mode and r['arm']==arm for r in clean_samples) for arm in ('dense','rank96')}
    print(f"{label}, without profiler: dense {means['dense']:.2f} → rank 96 {means['rank96']:.2f} ms/question; "
          f"mean visible tokens {lengths['dense']:.1f} → {lengths['rank96']:.1f}.")
    capped=sum(r['tokens'][-1]!=tokenizer.eos_token_id for r in clean_samples if r['mode']==mode and r['arm']=='rank96')
    if capped: print(f'{label}: {capped} measured generations reached the token cap; interpret timings with that in mind.')
print('Dense and rank-96 totals use their own generated sequences; differing lengths affect latency.')
# A concise setup total; individual setup operations are available in debug.jsonl.gz.
setup_ms=0
with gzip.open(DEBUG_PATH,'rt') as stream:
    for line in stream:
        record=json.loads(line)
        if record['kind']=='setup': setup_ms+=record['data']['cpu_wall_ms']
print(f'One-time setup and fitting: {setup_ms/60000:.1f} minutes. Raw values: {DEBUG_PATH}')

Optional inspection stays inside this notebook. Run this only when debugging a question:
```python
# All named submodules, token positions and individual CPU/CUDA event values for one question.
with gzip.open(DEBUG_PATH, 'rt') as stream:
    events = next(r['data'] for line in stream if (r := json.loads(line))['kind'] == 'events'
                  and r['data'][0]['mode'] == 'codi' and r['data'][0]['question_id'] == 0)
pd.DataFrame(events)  # filter name, phase or token_position here
```
The four-column table is also available as `bottleneck`; per-question rows as
`profile_samples`; clean baseline measurements as `clean_samples`. No CSV downloads are required.